In [ ]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    在模型处理之前修剪消息，只保留最近的几条消息以适应上下文窗口限制。
    
    修剪策略：
    - 如果消息总数 <= 3，不进行处理
    - 否则：保留第 1 条消息（通常是系统消息或首条消息）+ 最近 3-4 条消息
    - 这样可以保留重要的初始上下文，同时控制 token 数量
    """
    messages = state["messages"]

    # 如果消息数量不多，无需修剪
    if len(messages) <= 3:
        return None  # 返回 None 表示不修改状态

    # 第一条消息通常是系统消息或对话起点，需要保留
    first_msg = messages[0]
    
    # 根据消息总数的奇偶性决定保留最近 3 条还是 4 条消息
    # 这样可以保持 Human-AI 对话对的完整性（避免只保留单方面的消息）
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    
    # 组合新消息列表：第一条 + 最近的消息
    new_messages = [first_msg] + recent_messages

    # 返回更新指令：
    # 1. RemoveMessage(id=REMOVE_ALL_MESSAGES) - 清除所有现有消息
    # 2. *new_messages - 添加修剪后的消息
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


# 创建 Agent，将 trim_messages 中间件注册到 middleware 列表中
agent = create_agent(
    model,  # 使用之前定义的模型
    tools=[],  # 这里可以添加自定义工具
    middleware=[trim_messages],  # 注册消息修剪中间件
    checkpointer=InMemorySaver(),  # 启用短期记忆持久化
)

# 配置线程 ID，同一个 thread_id 的对话会共享记忆
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
# 模拟多轮对话，演示消息修剪效果

# 第一次调用：用户自我介绍
print("=== 第 1 轮：用户自我介绍 ===")
result1 = agent.invoke({"messages": "hi, my name is bob"}, config)
print(f"AI 回复：{result1['messages'][-1].content[:100]}...\n")

# 第二次调用：请求写诗
print("=== 第 2 轮：请求写关于猫的诗 ===")
result2 = agent.invoke({"messages": "write a short poem about cats"}, config)
print(f"AI 回复：{result2['messages'][-1].content[:100]}...\n")

# 第三次调用：继续请求
print("=== 第 3 轮：请求写关于狗的诗 ===")
result3 = agent.invoke({"messages": "now do the same but for dogs"}, config)
print(f"AI 回复：{result3['messages'][-1].content[:100]}...\n")

# 第四次调用：测试记忆 —— 此时早期的消息已被修剪，但 AI 仍应记得用户的名字
print("=== 第 4 轮：测试记忆（询问用户名字）===")
final_response = agent.invoke({"messages": "what's my name?"}, config)
final_response["messages"][-1].pretty_print()

### 消息修剪过程详解


**修剪策略可视化：**

```
初始消息列表（假设 7 条消息）:
[Msg0, Msg1, Msg2, Msg3, Msg4, Msg5, Msg6]
 │                     └───────┬───────┘
 │                             └─ 最近 3 条 (保留)
 └─ 第一条 (保留)

修剪后结果:
[Msg0, Msg4, Msg5, Msg6]
```

**关键点说明：**

1. **`@before_model` 装饰器**: 
   - 让 `trim_messages` 函数在模型处理之前执行
   - 每次 Agent 被调用时都会运行此中间件

2. **`RemoveMessage(id=REMOVE_ALL_MESSAGES)`**:
   - 这是一个特殊指令，告诉 LangGraph 清除状态中的所有消息
   - 然后重新添加修剪后的消息列表

3. **为什么保留第一条消息？**:
   - 第一条消息通常包含重要的初始上下文（如系统提示、用户自我介绍）
   - 保留它可以让 AI 记住对话的起点

4. **为什么根据奇偶性调整保留数量？**:
   - Human-AI 对话是成对的
   - 奇偶判断确保不会只保留单方面的消息（如只有 Human 消息而没有 AI 回复）

**预期输出：**
```
================================== Ai Message ==================================

Your name is Bob. You told me that earlier.
If you'd like me to call you a nickname or use a different name, just say the word.
```

即使中间的消息被修剪了，AI 仍然能记住用户的名字，因为第一条消息（包含自我介绍）被保留了。

# Usage 用法
通过在创建Agent的时候指定 **checkpointer** 来添加 **线程级别的持久化记忆**

【tips】

- Langchain的Agent将短期记忆作为Agent state的一部分进行管理
- 通过将这些信息存储在图的状态中，Agent可以在不同的线程中保持分离的同时，访问给定的完整上下文
- state通过checkpointer持久化到数据库（或者内存）中，以便线程能随时恢复
- 短期记忆在Agent被调用或一个步骤（如工具调用）完成时更新，并且在每个步骤开始时读取状态

In [ ]:
from langchain_openai import ChatOpenAI
import os

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3.2",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)


In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

# 模拟用户数据库
USER_DATABASE = {
    "user_123": {
        "name": "John Smith",
        "email": "john@example.com",
        "balance": 5000
    },
    "user_456": {
        "name": "Jane Doe", 
        "email": "jane@example.com",
        "balance": 3000
    }
}

@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """查询用户信息。从 Store 中获取 user_id，然后查询用户数据库返回详细信息。"""
    store = runtime.store
    
    # 从 store 中获取 user_id (使用 thread_id 作为 key)
    user_id_item: dict = store.get(("session",), "user_id")
    
    if not user_id_item:
        return "未找到用户 ID，请先登录"
    
    user_id = user_id_item.value
    
    # 查询数据库
    user = USER_DATABASE.get(user_id)
    
    if user:
        return f"用户信息：\n姓名：{user['name']}\n邮箱：{user['email']}\n余额：{user['balance']}"
    else:
        return f"未找到用户 ID: {user_id}"


@tool
def login(user_id: str, runtime: ToolRuntime) -> str:
    """用户登录，将 user_id 保存到 Store 中"""
    store = runtime.store
    
    # 将 user_id 保存到 store，使用 thread_id 作为 key
    store.put(("session",), "user_id", user_id) # session作为namespace
    
    return f"登录成功！用户 ID: {user_id} 已保存"

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

# Saver是存储对话历史的
# Store是存储结构化信息的

# 创建 Store 实例用于持久化用户 ID
store = InMemoryStore()

agent = create_agent(
    model = model, 
    tools=[get_user_info, login],
    checkpointer=InMemorySaver(),
    store=store,  # 绑定 store 到 agent
    system_prompt="多用工具查询，如果用户没有登录，引导用户先登录"
)


# 第一次调用：先登录，保存 user_id
print("=== 第一次调用：登录 ===")
result = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": "登录 user_123"
                   }]}, 
    {"configurable": {"thread_id": "1"}}
)
print(result["messages"][-1].content)

# 第二次调用：查询信息（无需再传 user_id，因为已经保存在 Store 中）
print("\n=== 第二次调用：查询信息 ===")
result = agent.invoke(    
    {"messages": [{"role": "user", "content": "我的信息是啥"}]},   
    {"configurable": {"thread_id": "1"}}
)

print(result["messages"][-1].content)

# 第三次调用：换一个 thread_id，应该查询不到（因为不同会话）
print("\n=== 第三次调用：新会话 (thread_id=2) ===")
result = agent.invoke(    
    {"messages": [{"role": "user", "content": "查询我信息"}]},   
    {"configurable": {"thread_id": "2"}}
)

print(result["messages"][-1].content)

### 【InMemorySaver & InMemoryStore 区别】

`InMemorySaver` 和 `InMemoryStore` 虽然都是存储在内存中的组件，但它们服务的**记忆类型**和**数据组织方式**有本质区别。

以下是两者的详细对比分析：

1. InMemorySaver (即 Checkpointer) —— 短期记忆
`InMemorySaver` 是一个 **检查点（Checkpointer）** 的内存实现，主要用于管理 **短期记忆（Short-term memory）** 。

*   **核心功能**：实现**线程级持久化（Thread-level persistence）**。它让 Agent 能够记住单个对话线程内的交互信息。
*   **组织单位**：通过 **`thread_id`** 来隔离数据。每个线程都有自己独立的状态快照。
*   **存储内容**：存储的是整个 `State`（状态），通常包括对话历史（messages）和自定义的状态字段。
*   **工作机制**：当 Agent 运行完一个步骤（如工具调用）时，`InMemorySaver` 会保存当前的快照；在下一次交互开始时，它会根据 `thread_id` 加载之前的快照以恢复上下文。
*   **局限性**：由于是“InMemory”，一旦程序重启，所有的短期记忆都会丢失。在生产环境中，通常会被 `SqliteSaver` 或 `PostgresSaver` 取代。

2. InMemoryStore —— 长期记忆
虽然来源中没有详细展开 `InMemoryStore` 的代码实现，但在 LangChain 的体系中（如前文对话所述），它是 **长期记忆（Long-term memory）** 的内存实现。

*   **核心功能**：实现 **跨线程持久化** 。它存储的信息不局限于某一个对话，而是可以被同一个用户的所有对话、甚至所有用户共享。
*   **组织单位**：使用 **命名空间/键模式（Namespace/Key pattern）** 。例如，你可以使用 `("user_123", "preferences")` 这样的路径来存储数据。
*   **存储内容**：存储的是结构化的知识或偏好信息（如“用户喜欢简洁的回答”），而不是原始的对话序列。
*   **工作机制**：在工具执行过程中，可以通过 `runtime.store` 显式地进行 `put`（写入）和 `get`（读取）操作。
*   **应用场景**：用于存储那些需要跨会话保留的背景知识、用户设置或历史总结。

3. 核心差异总结表

| 特性 | InMemorySaver (Checkpointer) | InMemoryStore (Store) |
| :--- | :--- | :--- |
| **记忆类型** | **短期记忆** (Thread-local) | **长期记忆** (Cross-thread) |
| **主要用途** | 维持当前对话的连贯性 | 存储跨会话的知识或用户偏好 |
| **检索索引** | `thread_id` | `Namespace` + `Key` |
| **自动程度** | **自动**。每步运行后由框架自动保存快照 | **手动**。通常需要在工具中显式调用读写 |
| **典型数据** | `[HumanMessage, AIMessage, ...]` | `{"language": "Chinese", "style": "brief"}` |
| **生产替代品** | `PostgresSaver`, `SqliteSaver` | `PostgresStore` |

**简单一句话总结**：
如果你想让 AI 记得 **“刚才我们聊了什么”**，请使用 `InMemorySaver`；

如果你想让 AI 记得 **“这个用户是谁，他有什么长期的喜好”** ，请使用 `InMemoryStore`。

# Customizing Agent Memory 
自定义Agent记忆

也就是添加一些自定义字段。

默认情况下，代理使用 **AgentState** 来管理短期记忆，具体通过 **messages** 键来管理 **对话历史**

可以通过扩展 AgentState 来扩展额外的字段。通过 state_schema 参数传递给 create_agent

In [ ]:
from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver
# from langgraph.store.memory import InMemoryStore
from langchain.messages import HumanMessage
import json

#? class AgentState(TypedDict, Generic[ResponseT]):
# AgentState 是 TypedDict 的子类，可以直接定义属性

class CustomAgentState(AgentState):
    user_id: str
    preferences: dict  # 用户偏好 dict
    
agent = create_agent(
    model,
    tools=[],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
    # context_schema=
)
# input: _InputAgentState | Command | None,
# 
res = agent.invoke(
    {
        "messages": [("human", "我的 user_id 是多少")],
        "user_id": "user_chasen",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}}
)

# 美化输出
output = {
    "messages": [
        {
            "type": msg.__class__.__name__,
            "content": msg.content[:150] + "..." if len(msg.content) > 150 else msg.content
        }
        for msg in res["messages"]
    ], # 这一整个是一个列表推导式
    "user_id": res.get("user_id"),
    "preferences": res.get("preferences")
}

print(json.dumps(output, indent=2, ensure_ascii=False))

简单说 **state_schema** 只是定义了 state 的结构，但你要自己负责把这些字段注入到 prompt 或 messages 中，Agent 才能"看到"并使用它们；或者这些字段也可以用于控制应用程序

# 常见模式
启动了短期记忆后，由于会存储聊天历史，这会导致长对话可能会超过LLM的上下文窗口。参加的解决方案有：

- Trim Messages：修剪信息，移除一些信息，比如说最早的5条
- Delete Messages：永久删除 langgraph 状态中的消息
- Summarize Messages：总结消息，总结早期消息，并且用摘要替换他们
- Custom Messages：自定义策略，比如说消息过滤

In [ ]:
from langchain.messages import AnyMessage, RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    在模型处理之前修剪消息，只保留最近的几条消息以适应上下文窗口限制。
    
    修剪策略：
    - 如果消息总数 <= 3，不进行处理
    - 否则：保留第 1 条消息（通常是系统消息或首条消息）+ 最近 3-4 条消息
    - 这样可以保留重要的初始上下文，同时控制 token 数量
    """
    messages = state["messages"]

    # 如果消息数量不多，无需修剪
    if len(messages) <= 3:
        return None  # 返回 None 表示不修改状态

    # 第一条消息通常是系统消息或对话起点，需要保留
    first_msg = messages[0]
    
    # 根据消息总数的奇偶性决定保留最近 3 条还是 4 条消息
    # 这样可以保持 Human-AI 对话对的完整性（避免只保留单方面的消息）
    recent_messages: list[AnyMessage] = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    
    # 组合新消息列表：第一条 + 最近的消息
    new_messages = [first_msg] + recent_messages

    # 返回更新指令：
    # 1. RemoveMessage(id=REMOVE_ALL_MESSAGES) - 清除所有现有消息
    # 2. *new_messages - 添加修剪后的消息
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages # 解包 list[AnyMessage]
        ]
    }
    
# 创建 Agent，将 trim_messages 中间件注册到 middleware 列表中
agent = create_agent(
    model,  # 使用之前定义的模型
    tools=[],  # 这里可以添加自定义工具
    middleware=[trim_messages],  # 注册消息修剪中间件
    checkpointer=InMemorySaver(),  # 启用短期记忆持久化
)

# 配置线程 ID，同一个 thread_id 的对话会共享记忆
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# 模拟多轮对话，演示消息修剪效果

# 第一次调用：用户自我介绍
print("=== 第 1 轮：用户自我介绍 ===")
result1 = agent.invoke({"messages": "hi, my name is bob"}, config)
print(f"AI 回复：{result1['messages'][-1].content[:100]}...\n")

# 第二次调用：请求写诗
print("=== 第 2 轮：请求写关于猫的诗 ===")
result2 = agent.invoke({"messages": "write a short poem about cats"}, config)
print(f"AI 回复：{result2['messages'][-1].content[:100]}...\n")

# 第三次调用：继续请求
print("=== 第 3 轮：请求写关于狗的诗 ===")
result3 = agent.invoke({"messages": "now do the same but for dogs"}, config)
print(f"AI 回复：{result3['messages'][-1].content[:100]}...\n")

# 第四次调用：测试记忆 —— 此时早期的消息已被修剪，但 AI 仍应记得用户的名字
print("=== 第 4 轮：测试记忆（询问用户名字）===")
final_response = agent.invoke({"messages": "what's my name?"}, config)
final_response["messages"][-1].pretty_print()

# 总结消息
Summarize Messages

修剪消息和删除消息可能会丢失一些信息，所以可以使用LLM来总结消息 Compact

要在Agent中总结消息历史，使用Middleware的SummarizationMiddleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
import os

checkpointer = InMemorySaver()

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3.2",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)

agent = create_agent(
    model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 40), # 到达tokens为20就总结
            keep=("messages", 1) # 保留一条消息
        )
        # ContextSize 是一个元组
    ],
    checkpointer=checkpointer
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我叫Chasen"}, config)["messages"][-1].pretty_print()
agent.invoke({"messages": "你说现有鸡还是先有蛋"}, config)["messages"][-1].pretty_print()
agent.invoke({"messages": "我是一个计算机的学生"}, config)["messages"][-1].pretty_print()
agent.invoke({"messages": "我喜欢hhy"}, config)["messages"][-1].pretty_print()

final_res = agent.invoke({"messages": "我刚刚和你说了啥?然后我叫什么名字"}, config)

final_res["messages"][-1].pretty_print()

================================== Ai Message ==================================

你好，Chasen！很高兴认识你！😊  
这是一个很特别的名字，有什么特别的含义或者故事吗？  
无论你想聊天、提问，还是需要帮助，我都很乐意为你提供支持。期待和你的交流！
================================== Ai Message ==================================

这是一个经典的哲学与生物学问题，从不同角度会有不同的答案：

**1. 生物学进化角度：**  
现代科学认为“蛋”的出现早于“鸡”。因为鸡属于鸟类，而鸟类是由恐龙进化而来的。在进化过程中，某种接近鸡的原始鸟类（或恐龙）产下了一枚含有基因突变的蛋，孵化出了第一只符合现代鸡定义的生物。因此，**先有蛋**（由接近鸡的物种产下的蛋中诞生了第一只鸡）。

**2. 逻辑循环角度：**  
如果严格定义“鸡生的蛋叫鸡蛋”和“鸡蛋孵出的叫鸡”，则会陷入循环。但若定义“含有鸡的胚胎的蛋叫鸡蛋”，则第一只鸡来自某个非鸡生物产下的蛋（蛋内发生了关键基因突变），因此**先有蛋**。

**3. 哲学与文化隐喻：**  
这个问题常被用来讨论“因果困境”，象征无限循环的因果关系。许多文化中会用类似问题引发对生命起源、创造论的思考。

**简单总结：**  
目前科学界更支持“先有蛋”，因为进化是通过遗传变异在生殖细胞（如蛋中的胚胎）中实现的，突变发生在蛋形成的阶段，从而诞生了新物种。

你对这个问题的哪个方面更感兴趣呢？ 😊
================================== Ai Message ==================================

你好，计算机专业的同学！既然你是学计算机的，那我们可以用一些更“代码友好”或“系统思维”的方式来重新审视“先有鸡还是先有蛋”这个问题。

### 1. **从“定义”与“类型”的角度看（编程思维）**
这本质上是一个**类型定义（Class）和实例化（Instantiation）的先后问题**。

*   **如果我们将“鸡”定义为一个类（`class Chicken`）：**
    *   那么第一个符合 

# Access Memory 访问记忆
可以通过多种方式访问和修改Agent的state，也就是短期记忆

## Tools 
通过工具中传入的 `runtime: ToolRuntime` 在tool中访问短期记忆

runtime 参数在工具签名中被隐藏（因此模型看不到它），但工具可以通过它访问状态。


In [1]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI
import os

# 自定义一个State的Schema
class CustomState(AgentState):
    user_id: str
    
# 通过tool来读取state然后返回结果，这样state可以被Agent看到    
    
def get_user_info(runtime: ToolRuntime) -> str:
    """获取用户信息"""
    user_id = runtime.state["user_id"]
    return "用户的Id是" + user_id if user_id else "查不到用户消息"

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3.2",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)

agent = create_agent(
    model,
    tools=[get_user_info],
    state_schema=CustomState,
)

res = agent.invoke({
    "messages": "查找一下用户信息",
    "user_id": "Chasen"
})

print(res["messages"][-1].content) # 

根据查询结果，用户的ID是：**Chasen**


## 通过Tools写入短期记忆

要在执行过程中修改代理的短期记忆（状态）；这有助于持久化中间结果或使信息可供后续工具或提示使用

In [ ]:
from langchain_openai import ChatOpenAI
import os 

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)

In [13]:
from langchain.agents import create_agent, AgentState
from langchain.tools import ToolRuntime, tool
from langchain_core.runnables import RunnableConfig
from langchain.messages import ToolMessage
from langgraph.types import Command
from pydantic import BaseModel

#? 自定义的state继承 AgentState
class CustomState(AgentState): 
    user_name: str

#? 自定义的Context继承 BaseModel
class CustomContext(BaseModel):
    user_id: str
    
# 更新user_info
@tool("查到用户信息")
def update_user_info(
    runtime: ToolRuntime[CustomContext, CustomState]
) -> Command:
    """更新用户信息"""
    # user_id 存放在上下文Context中
    user_id = runtime.context.user_id # 这是个class, 用.来取属性
    name = "Chasen" if user_id else "没法查到用户信息"
    # 在context中查到user_id，然后更新到state中
    return Command(
        update={
            "user_name": name,
            # 更新消息历史
            "messages": [
                ToolMessage(
                    "成功查到用户信息",
                    tool_call_id=runtime.tool_call_id # 工具的id
                )
            ]
        }
    )
    
    
# 打招呼tool
@tool
def greet(
    runtime: ToolRuntime[CustomContext, CustomState]
) -> Command | str:
    """Use this to greet the user once you found their info."""
    # 从state中获取user_name
    user_name = runtime.state.get("user_name", None)
    if user_name is None:
        # 通过Command来传入一个ToolMessage提示LLM没有查到信息
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        "Please call the 'update_user_info' tool it will get and update the user's name.",
                        tool_call_id=runtime.tool_call_id
                    )
                ]
            }
        )
    return f"你好 {user_name}!"

agent = create_agent(
    model,
    tools=[greet, update_user_info],
    state_schema=CustomState,
    context_schema=CustomContext,
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "和用户打招呼"}]},
    context=CustomContext(user_id="Chasen"),
    stream_mode="values"
):
    chunk["messages"][-1].pretty_print()


================================ Human Message =================================

和用户打招呼
================================== Ai Message ==================================
Tool Calls:
  查到用户信息 (019cf9941704fcf1d191f0c68c830c6b)
 Call ID: 019cf9941704fcf1d191f0c68c830c6b
  Args:
================================= Tool Message =================================
Name: 查到用户信息

成功查到用户信息
================================== Ai Message ==================================
Tool Calls:
  greet (019cf9941d54757b5f095a401b5cf4ee)
 Call ID: 019cf9941d54757b5f095a401b5cf4ee
  Args:
================================= Tool Message =================================
Name: greet

你好 Chasen!
================================== Ai Message ==================================

你好 Chasen!


如果交换写成 ToolRuntime[CustomState, CustomContext]，则 runtime.context 实际上会变成 State 类型，而 runtime.state 会变成 Context 类型，导致类型不匹配和逻辑错误. 【真的吗？】